#1. Persiapan Library

In [ ]:
# @title 1a. Install kagglehub & mount Google Drive
!pip install -q kagglehub

from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# @title 1b. Import Library

import os
import random

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm

sns.set_style("whitegrid")


def set_seed(seed: int = 42):
    """Set semua random seed agar hasil reproducible."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Menggunakan device: {DEVICE}")

#2. Download Dataset

In [ ]:
# @title APTOS 2019 (224x224)
aptos_path = kagglehub.dataset_download(
    "sovitrath/diabetic-retinopathy-224x224-2019-data"
)
print("Path to dataset files:", aptos_path)

# struktur dataset ini: <aptos_path>/train.csv + <aptos_path>/colored_images/<kelas>/<id_code>.png
# struktur csv ini: "id_code" dan "diagnosis"
!ls "{aptos_path}"

In [ ]:
# @title DDR
ddr_path = kagglehub.dataset_download("mariaherrerot/ddrdataset")
print("Path to dataset files:", ddr_path)

# struktur dataset ini: <ddr_path>/DR_grading.csv + <ddr_path>/DR_grading/DR_grading/<id_code>.jpg
# struktur csv ini: "id_code" dan "diagnosis"
!ls "{ddr_path}"

In [ ]:
# @title DR_Resized
drr_path = kagglehub.dataset_download("tanlikesmath/diabetic-retinopathy-resized")
print("Path to dataset files:", drr_path)

# struktur dataset ini: <drr_path>/trainLabels.csv + <drr_path>/resized_train/resized_train/<image>.jpeg
# struktur csv ini: "image" dan "level"
!ls "{drr_path}"

In [ ]:
# @title Dataset Lokal (Google Drive ke disk lokal Colab)
import glob
import shutil

LOCAL_DR_DIR_DRIVE = "/content/drive/MyDrive/BayesianCNN/Dataset/Lokal"
LOCAL_DR_DIR = "/content/data/local_raw"


def copy_drive_to_local(src_dir, dst_dir):
    os.makedirs(dst_dir, exist_ok=True)
    all_files = [
        f
        for f in glob.glob(os.path.join(src_dir, "**", "*"), recursive=True)
        if os.path.isfile(f)
    ]
    copied, skipped = 0, 0
    for src_fp in tqdm(all_files, desc="Copy Drive -> lokal Colab"):
        rel_path = os.path.relpath(src_fp, src_dir)
        dst_fp = os.path.join(dst_dir, rel_path)
        os.makedirs(os.path.dirname(dst_fp), exist_ok=True)
        if os.path.exists(dst_fp) and os.path.getsize(dst_fp) == os.path.getsize(
            src_fp
        ):
            skipped += 1
            continue
        shutil.copy2(src_fp, dst_fp)
        copied += 1
    print(
        f"Selesai copy: {copied} file baru, {skipped} file sudah ada (di-skip, idempotent kalau re-run)."
    )


copy_drive_to_local(LOCAL_DR_DIR_DRIVE, LOCAL_DR_DIR)

print("Isi folder lokal:")
for cls in ["No_DR", "Mild", "Moderate", "Severe", "Proliferate_DR"]:
    n = len(glob.glob(os.path.join(LOCAL_DR_DIR, cls, "*")))
    print(f"  {cls}: {n} file")

#3. Build Unified Dataframe Setiap Dataset

In [ ]:
# @title DF Dataset APTOS + Lokal (sama-sama struktur class_folders)
CLASS_ALIASES = {
    "No_DR": ["No_DR", "0"],
    "Mild": ["Mild", "1"],
    "Moderate": ["Moderate", "2"],
    "Severe": ["Severe", "3"],
    "Proliferate_DR": ["Proliferate_DR", "4"],
}


def load_class_folder_df(root_dir, source_name):
    rows = []
    for label, aliases in CLASS_ALIASES.items():
        folder_path = next(
            (
                os.path.join(root_dir, a)
                for a in aliases
                if os.path.isdir(os.path.join(root_dir, a))
            ),
            None,
        )
        if folder_path is None:
            continue
        for fp in glob.glob(os.path.join(folder_path, "*")):
            if fp.lower().endswith((".png", ".jpg", ".jpeg")):
                rows.append({"filepath": fp, "label": label, "source": source_name})
    return pd.DataFrame(rows)


aptos_df = load_class_folder_df(os.path.join(aptos_path, "colored_images"), "aptos")
print("Aptos")
print(aptos_df["label"].value_counts().sort_index())
print("\n")

local_df = load_class_folder_df(LOCAL_DR_DIR, "local")
print("Local")
print(local_df["label"].value_counts().sort_index())
print("\n")

In [ ]:
# @title DF Dataset DDR
ddr_csv_path = os.path.join(ddr_path, "DR_grading.csv")
ddr_meta = pd.read_csv(ddr_csv_path)

# Mapping label
ddr_label_map = {0: "No_DR", 1: "Mild", 2: "Moderate", 3: "Severe", 4: "Proliferate_DR"}

ddr_rows = []
# DDR grading folder path
ddr_img_dir = os.path.join(ddr_path, "DR_grading", "DR_grading")

for _, row in ddr_meta.iterrows():
    img_name = row["id_code"]
    # DDR images path
    img_path = os.path.join(ddr_img_dir, img_name)
    if not os.path.exists(img_path):
        # Tambahkan ekstensi jika tidak ada
        if os.path.exists(img_path + ".jpg"):
            img_path += ".jpg"
        elif os.path.exists(img_path + ".png"):
            img_path += ".png"
        elif os.path.exists(img_path + ".jpeg"):
            img_path += ".jpeg"

    if os.path.exists(img_path):
        label_idx = row["diagnosis"]
        if label_idx in ddr_label_map:
            ddr_rows.append(
                {
                    "filepath": img_path,
                    "label": ddr_label_map[label_idx],
                    "source": "ddr",
                }
            )

ddr_df = pd.DataFrame(ddr_rows)
print(f"Total data DDR yang valid: {len(ddr_df)}")
print(ddr_df["label"].value_counts())

In [ ]:
# @title DF Dataset DR_Resized
drr_csv_path = os.path.join(drr_path, "trainLabels_cropped.csv")
drr_meta = pd.read_csv(drr_csv_path)

# Mapping label DR-Resized (level 0-4)
drr_label_map = {0: "No_DR", 1: "Mild", 2: "Moderate", 3: "Severe", 4: "Proliferate_DR"}

drr_rows = []
# Path ke folder DR-Resized
drr_img_dir = os.path.join(drr_path, "resized_train_cropped", "resized_train_cropped")

for _, row in drr_meta.iterrows():
    img_name = str(row["image"]) + ".jpeg"
    img_path = os.path.join(drr_img_dir, img_name)

    if os.path.exists(img_path):
        level = row["level"]
        if level in drr_label_map:
            drr_rows.append(
                {
                    "filepath": img_path,
                    "label": drr_label_map[level],
                    "source": "dr_resized",
                }
            )

drr_final_df = pd.DataFrame(drr_rows)
print(f"Total data DR-Resized valid: {len(drr_final_df)}")
print(drr_final_df["label"].value_counts())

#4. Gabungkan Dataset

In [ ]:
full_df = pd.concat([aptos_df, local_df, ddr_df], ignore_index=True)
full_df = full_df[full_df["filepath"].apply(os.path.exists)].reset_index(drop=True)
df_shuffled = full_df.sample(frac=1, random_state=42).reset_index(drop=True)
print("| combined (valid files):", len(df_shuffled))
print(df_shuffled["label"].value_counts().sort_index())

#5. Balancing Dataset (Median-based Oversampling & Undersampling)

In [ ]:
# 1. Hitung jumlah sampel per kelas
counts = full_df["label"].value_counts()
median_samples = int(counts.median())
print(f"Distribusi asli:\n{counts}\n")
print(f"Target balancing (Median): {median_samples} sampel per kelas")

balanced_list = []

for label in counts.index:
    df_class = full_df[full_df["label"] == label]
    n_class = len(df_class)

    if n_class > median_samples:
        # Undersampling untuk kelas mayoritas
        df_resampled = df_class.sample(median_samples, random_state=42)
        print(f" - {label}: Undersampling dari {n_class} ke {median_samples}")
    else:
        # Oversampling untuk kelas minoritas
        # replace=True memungkinkan pengambilan sampel berulang
        df_resampled = df_class.sample(median_samples, replace=True, random_state=42)
        print(f" - {label}: Oversampling dari {n_class} ke {median_samples}")

    balanced_list.append(df_resampled)

balanced_df = pd.concat(balanced_list).reset_index(drop=True)

print("\nDistribusi label setelah balancing:")
print(balanced_df["label"].value_counts())


# Update df_shuffled dengan data yang sudah balanced
df_shuffled = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

#6. Preprocessor

In [ ]:
import cv2

cv2.setNumThreads(0)


def _crop_tight_retina(img, thresh=10):
    """
    Memotong gambar tepat di batas terluar retina (tight bounding box)
    tanpa menambahkan padding buatan.
    """
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    mask = gray > thresh

    if not np.any(mask):
        return img  # Fallback jika gambar hitam murni

    coords = np.argwhere(mask)
    y0, x0 = coords.min(axis=0)
    y1, x1 = coords.max(axis=0)

    # Memastikan bounding box valid
    if (y1 - y0) < 10 or (x1 - x0) < 10:
        return img

    return img[y0:y1, x0:x1]


def _denoise(img, ksize=3):
    return cv2.medianBlur(img, ksize)


def _clahe_green_channel(img_rgb, clip_limit=2.5, tile=(8, 8)):
    r, g, b = cv2.split(img_rgb)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile)
    g_eq = clahe.apply(g)
    return cv2.merge((r, g_eq, b))


def _local_average_subtract(img, sigma_fraction=30):
    sigma = max(img.shape[0], img.shape[1]) / sigma_fraction
    blurred = cv2.GaussianBlur(img, (0, 0), sigma)
    out = cv2.addWeighted(img, 4, blurred, -4, 128)
    return out


def _circular_mask(img, radius_fraction=0.93):
    """
    Membuat topeng lingkaran rapi untuk membuang artefak tepi
    dan memaksa latar belakang di luar lingkaran menjadi hitam murni (0,0,0).
    """
    h, w = img.shape[:2]
    mask = np.zeros((h, w), dtype=np.uint8)
    center = (w // 2, h // 2)
    radius = int(min(h, w) / 2 * radius_fraction)

    cv2.circle(mask, center, radius, 255, -1)
    out = cv2.bitwise_and(img, img, mask=mask)
    return out


def preprocess_image(img_rgb, img_size=224, radius_fraction=0.86, debug_stats=None):
    """
    Pipeline preprocessing:
    1. Tight Crop -> 2. Direct Resize (Memenuhi Frame) -> 3. Denoise
    -> 4. CLAHE -> 5. Ben Graham -> 6. Circular Mask (Smooth Edge)
    """
    original = img_rgb.copy()
    try:
        # 1. Potong pas pada area retina (membuang margin hitam bawaan)
        cropped = _crop_tight_retina(img_rgb, thresh=10)

        # 2. Resize langsung ke (224, 224) agar retina memenuhi bingkai tanpa padding abu-abu
        img_rgb = cv2.resize(
            cropped, (img_size, img_size), interpolation=cv2.INTER_AREA
        )

        # 3. Denoising
        img_rgb = _denoise(img_rgb, ksize=3)

        # 4. CLAHE Enhancement (Green Channel)
        img_rgb = _clahe_green_channel(img_rgb, clip_limit=2.5, tile=(8, 8))

        # 5. Normalisasi Iluminasi (Ben Graham Subtraction)
        img_rgb = _local_average_subtract(img_rgb, sigma_fraction=30)

        # 6. Circular Masking (93%) untuk merapikan tepi dan memastikan background 100% hitam
        img_rgb = _circular_mask(img_rgb, radius_fraction=radius_fraction)

    except Exception as e:
        img_rgb = cv2.resize(
            original, (img_size, img_size), interpolation=cv2.INTER_AREA
        )
        if debug_stats is not None:
            debug_stats["failed"] = debug_stats.get("failed", 0) + 1
            debug_stats.setdefault("errors", []).append(str(e))

    return img_rgb

In [ ]:
# @title Sanity check preprocessing on sample images

import glob


def _load_rgb(path):
    img = cv2.imread(path)
    if img is None:
        raise ValueError(f"Gagal membaca gambar: {path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def visualize_samples(df, source_name, num_samples=5):
    sample_paths = (
        df[df["source"] == source_name]["filepath"].head(num_samples).tolist()
    )
    if not sample_paths:
        print(f"Tidak ada sampel untuk sumber: {source_name}")
        return

    fig, axes = plt.subplots(2, len(sample_paths), figsize=(4 * len(sample_paths), 8))
    fig.suptitle(
        f"Sanity Check Preprocessing: Source {source_name.upper()}", fontsize=16
    )

    for i, p in enumerate(sample_paths):
        raw = _load_rgb(p)
        proc = preprocess_image(raw)

        # Tampilkan Raw
        axes[0, i].imshow(raw)
        axes[0, i].set_title("Original")
        axes[0, i].axis("off")

        # Tampilkan Preprocessed
        axes[1, i].imshow(proc)
        axes[1, i].set_title("Preprocessed")
        axes[1, i].axis("off")

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()


# Visualisasi untuk tiap sumber
for src in df_shuffled["source"].unique():
    visualize_samples(df_shuffled, src)

#7. Split Data (Stratified by Label & Source)

In [ ]:

# 1. Buat kolom dummy untuk stratifikasi gabungan
df_shuffled["stratify_key"] = (
    df_shuffled["label"].astype(str) + "_" + df_shuffled["source"].astype(str)
)

# 2. Penanganan sampel langka: Jika suatu kombinasi label_source < 3 sampel (tidak cukup)
counts = df_shuffled["stratify_key"].value_counts()
rare_keys = counts[counts < 3].index
df_shuffled["stratify_key"] = df_shuffled["stratify_key"].apply(
    lambda x: x if x not in rare_keys else "rare_combined"
)

# 3. Split pertama (Train vs Temp: 70/30)
train_df, temp_df = train_test_split(
    df_shuffled, test_size=0.30, stratify=df_shuffled["stratify_key"], random_state=42
)

# 4. Split kedua (Val vs Test: 50/50 dari 30%)
temp_counts = temp_df["stratify_key"].value_counts()
rare_temp_keys = temp_counts[temp_counts < 2].index

if len(rare_temp_keys) > 0:
    # Jika < 2, gunakan stratifikasi berdasarkan label untuk split kedua
    strat_col = temp_df["label"]
else:
    strat_col = temp_df["stratify_key"]

val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=strat_col, random_state=42
)

# Bersihkan kolom dummy
train_df = train_df.drop(columns=["stratify_key"]).reset_index(drop=True)
val_df = val_df.drop(columns=["stratify_key"]).reset_index(drop=True)
test_df = test_df.drop(columns=["stratify_key"]).reset_index(drop=True)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# Visualisasi distribusi untuk verifikasi
print("\nDistribusi Source di Train set:")
print(train_df["source"].value_counts(normalize=True).round(3))

print("\nDistribusi Label di Train set:")
print(train_df["label"].value_counts(normalize=True).round(3))

#8. Persiapan Dataset Pelatihan

In [ ]:
# @title Precompute preprocessing cache
import hashlib

CACHE_DIR = "/content/preprocess_cache"
os.makedirs(CACHE_DIR, exist_ok=True)


def _cache_path(filepath, img_size=224):
    h = hashlib.md5(f"{filepath}_{img_size}".encode()).hexdigest()
    return os.path.join(CACHE_DIR, f"{h}.png")


def precompute_cache(df, img_size=224):
    """Jalankan preprocess_image SEKALI utk tiap gambar unik, simpan hasilnya sbg
    PNG 224x224 di CACHE_DIR. Idempotent: kalau file cache sudah ada, dilewati
    (aman dipanggil ulang / lanjut kalau sesi Colab keputus di tengah jalan)."""
    n_done, n_skip, n_fail = 0, 0, 0
    for fp in tqdm(df["filepath"].unique(), desc="Precompute cache"):
        cpath = _cache_path(fp, img_size)
        if os.path.exists(cpath):
            n_skip += 1
            continue
        img_bgr = cv2.imread(fp)
        if img_bgr is None:
            print(f"WARNING: gagal load {fp}, dilewati")
            n_fail += 1
            continue
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_proc = preprocess_image(img_rgb, img_size=img_size)
        cv2.imwrite(cpath, cv2.cvtColor(img_proc, cv2.COLOR_RGB2BGR))
        n_done += 1
    print(
        f"Selesai: {n_done} gambar baru di-cache, {n_skip} sudah ada (skip), {n_fail} gagal."
    )


precompute_cache(
    df_shuffled, img_size=224
)  # sekali jalan utk seluruh dataset (train+val+test)

In [ ]:
# @title Custom PyTorch Dataset (dgn cache preprocessing)


class DRDataset(Dataset):
    LABEL_TO_IDX = {
        "No_DR": 0,
        "Mild": 1,
        "Moderate": 2,
        "Severe": 3,
        "Proliferate_DR": 4,
    }

    def __init__(self, df, img_size=224, transform=None, use_cache=True):
        self.df = df.reset_index(drop=True)
        self.img_size = img_size
        self.transform = transform
        self.use_cache = use_cache

    def __len__(self):
        return len(self.df)

    def _load_processed(self, filepath):
        if self.use_cache:
            cpath = _cache_path(filepath, self.img_size)
            if os.path.exists(cpath):
                img_bgr = cv2.imread(cpath)
                return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        # cache miss
        img_bgr = cv2.imread(filepath)
        if img_bgr is None:
            raise ValueError(f"Gagal membaca gambar: {filepath}")
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        return preprocess_image(img_rgb, img_size=self.img_size)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_proc = self._load_processed(row["filepath"])
        img_pil = Image.fromarray(img_proc)

        if self.transform is not None:
            img_tensor = self.transform(img_pil)
        else:
            img_tensor = transforms.ToTensor()(img_pil)

        label_idx = self.LABEL_TO_IDX[row["label"]]
        return img_tensor, label_idx

In [ ]:
# @title Transforms (augmentation + normalisasi) & instansiasi Dataset
IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# augmentasi hanya untuk train; val/test cuma normalisasi (evaluasi harus deterministik)
train_transform = transforms.Compose(
    [
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(degrees=20),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)

eval_transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)

train_dataset = DRDataset(train_df, img_size=IMG_SIZE, transform=train_transform)
val_dataset = DRDataset(val_df, img_size=IMG_SIZE, transform=eval_transform)
test_dataset = DRDataset(test_df, img_size=IMG_SIZE, transform=eval_transform)

# sanity check satu sampel
sample_img, sample_label = train_dataset[0]
print(
    "Sample image tensor:", sample_img.shape, sample_img.dtype, "| label:", sample_label
)

In [ ]:
# @title DataLoader + imbalance handling
from torch.utils.data import WeightedRandomSampler

# WeightedRandomSampler: kelas minoritas (Severe, Proliferate_DR) di-oversample
# secara acak per-batch, supaya training tidak bias ke No_DR/Moderate yang dominan
class_counts = train_df["label"].value_counts()
class_weight_map = {cls: 1.0 / count for cls, count in class_counts.items()}
sample_weights = train_df["label"].map(class_weight_map).values

train_sampler = WeightedRandomSampler(
    weights=sample_weights, num_samples=len(sample_weights), replacement=True
)

BATCH_SIZE = 64
NUM_WORKERS = 0

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=train_sampler,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print(
    f"Train data: {len(train_dataset)} | Val data: {len(val_dataset)} | Test data: {len(test_dataset)}"
)
print(
    f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}"
)

imgs, labels = next(iter(train_loader))
print("Batch images:", imgs.shape, "| Batch labels:", labels.shape)
print(
    "Distribusi label dalam 1 batch (harusnya lebih merata krn sampler):",
    labels.bincount(),
)

#9. Bayesian CNN (DenseNet-121 backbone + MC Dropout)

In [ ]:
# @title BCNN model & MC Dropout helpers
NUM_CLASSES = 5
DROPOUT_P = 0.3


class BayesianDRNet(nn.Module):
    """DenseNet-121 pretrained sebagai feature extractor + classifier head dengan
    Dropout."""

    def __init__(self, num_classes=NUM_CLASSES, dropout_p=DROPOUT_P, pretrained=True):
        super().__init__()
        weights = models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = models.densenet121(weights=weights)
        in_features = backbone.classifier.in_features
        backbone.classifier = (
            nn.Identity()
        )  # buang classifier bawaan, hanya ambil fitur
        self.backbone = backbone
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout_p),
            nn.Linear(in_features, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_p),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        features = self.backbone(x)
        logits = self.classifier(features)
        return logits


def enable_mc_dropout(model: nn.Module):
    """Set model ke eval mode (BatchNorm pakai running stats), TAPI paksa semua
    layer nn.Dropout tetap aktif -> kunci MC Dropout saat inference."""
    model.eval()
    for module in model.modules():
        if isinstance(module, nn.Dropout):
            module.train()


@torch.no_grad()
def mc_dropout_predict(model: nn.Module, x: torch.Tensor, T: int = 25):
    """Jalankan T forward pass dengan dropout aktif, lalu menghitung:
    - mean_probs            : rata-rata probabilitas softmax (prediksi akhir)
    - predictive_entropy    : total uncertainty (aleatoric + epistemic)
    - aleatoric_entropy     : rata-rata entropy tiap sample (uncertainty dari data)
    - epistemic_uncertainty : mutual information/BALD = predictive - aleatoric,
      merepresentasikan seberapa 'model tidak yakin karena kurang belajar'
    """
    enable_mc_dropout(model)
    probs_samples = []
    for _ in range(T):
        logits = model(x)
        probs = F.softmax(logits, dim=1)
        probs_samples.append(probs.unsqueeze(0))
    probs_samples = torch.cat(probs_samples, dim=0)  # (T, B, num_classes)

    mean_probs = probs_samples.mean(dim=0)  # (B, num_classes)
    eps = 1e-12
    predictive_entropy = -(mean_probs * torch.log(mean_probs + eps)).sum(dim=1)  # (B,)
    per_sample_entropy = -(probs_samples * torch.log(probs_samples + eps)).sum(
        dim=2
    )  # (T, B)
    aleatoric_entropy = per_sample_entropy.mean(dim=0)  # (B,)
    epistemic_uncertainty = predictive_entropy - aleatoric_entropy  # (B,) >= 0

    return {
        "mean_probs": mean_probs,
        "predictive_entropy": predictive_entropy,
        "aleatoric_entropy": aleatoric_entropy,
        "epistemic_uncertainty": epistemic_uncertainty,
    }


model = BayesianDRNet(num_classes=NUM_CLASSES, dropout_p=DROPOUT_P, pretrained=True).to(
    DEVICE
)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model dibuat. Trainable params: {n_params:,}")

#10. Pelatihan Model

In [ ]:
# @title Loss, Optimizer & Scheduler
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=5e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=5
)

In [ ]:
# @title Training loop
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []
    pbar = tqdm(loader, desc="Train", leave=False)
    for imgs, labels in pbar:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        preds = logits.argmax(dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())
        pbar.set_postfix(loss=loss.item())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_qwk = cohen_kappa_score(all_labels, all_preds, weights="quadratic")
    return epoch_loss, epoch_acc, epoch_qwk


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """Evaluasi standar (deterministik, dropout OFF) -> dipakai tiap epoch untuk
    validasi."""
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []
    pbar = tqdm(loader, desc="Val", leave=False)
    for imgs, labels in pbar:
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        loss = criterion(logits, labels)
        running_loss += loss.item() * imgs.size(0)
        preds = logits.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        pbar.set_postfix(loss=loss.item())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_qwk = cohen_kappa_score(all_labels, all_preds, weights="quadratic")
    return epoch_loss, epoch_acc, epoch_qwk


N_EPOCHS = 50
PATIENCE = 10  # early stopping
CKPT_PATH = "/content/BCNN.pt"

history = {
    "train_loss": [],
    "train_acc": [],
    "train_qwk": [],
    "val_loss": [],
    "val_acc": [],
    "val_qwk": [],
}

best_val_qwk = -1.0
epochs_no_improve = 0

for epoch in range(1, N_EPOCHS + 1):
    train_loss, train_acc, train_qwk = train_one_epoch(
        model, train_loader, optimizer, criterion, DEVICE
    )
    val_loss, val_acc, val_qwk = evaluate(model, val_loader, criterion, DEVICE)
    scheduler.step(val_qwk)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["train_qwk"].append(train_qwk)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["val_qwk"].append(val_qwk)

    current_lr = optimizer.param_groups[0]["lr"]
    print(
        f"Epoch {epoch:02d}/{N_EPOCHS} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} train_qwk={train_qwk:.4f} | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_qwk={val_qwk:.4f} | lr={current_lr:.2e}"
    )

    if val_qwk > best_val_qwk:
        best_val_qwk = val_qwk
        epochs_no_improve = 0
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "epoch": epoch,
                "val_qwk": val_qwk,
            },
            CKPT_PATH,
        )
        print(f"  -> checkpoint disimpan (val_qwk={val_qwk:.4f})")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(
                f"Early stopping di epoch {epoch} (tidak ada peningkatan val_qwk selama {PATIENCE} epoch)."
            )
            break

print(f"\nTraining selesai. Best val QWK: {best_val_qwk:.4f} (checkpoint: {CKPT_PATH})")

In [ ]:
# @title Plot Training Curves
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
metrics = [("loss", "Loss"), ("acc", "Accuracy"), ("qwk", "Quadratic Weighted Kappa")]
for ax, (key, title) in zip(axes, metrics):
    ax.plot(history[f"train_{key}"], label="train")
    ax.plot(history[f"val_{key}"], label="val")
    ax.set_title(title)
    ax.set_xlabel("epoch")
    ax.legend()
plt.tight_layout()
plt.show()

#11. Evaluasi Performa

In [ ]:
# @title Load Model & Config
def load_best_checkpoint(model, ckpt_path, device):
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    print(
        f"Checkpoint dimuat dari epoch {ckpt['epoch']} (val_qwk={ckpt['val_qwk']:.4f})"
    )
    return model


model = load_best_checkpoint(model, CKPT_PATH, DEVICE)

T_MC = 25  # jumlah forward pass MC Dropout saat inference

In [ ]:
# @title Evaluate on Test Set
all_preds = []
all_labels = []
all_mean_probs = []
all_predictive_entropy = []
all_aleatoric_entropy = []
all_epistemic_uncertainty = []

for imgs, labels in tqdm(test_loader, desc="MC Dropout inference (test)"):
    imgs = imgs.to(DEVICE)
    result = mc_dropout_predict(model, imgs, T=T_MC)

    preds = result["mean_probs"].argmax(dim=1).cpu().numpy()
    all_preds.extend(preds)
    all_labels.extend(labels.numpy())
    all_mean_probs.append(result["mean_probs"].cpu().numpy())
    all_predictive_entropy.extend(result["predictive_entropy"].cpu().numpy())
    all_aleatoric_entropy.extend(result["aleatoric_entropy"].cpu().numpy())
    all_epistemic_uncertainty.extend(result["epistemic_uncertainty"].cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_mean_probs = np.concatenate(all_mean_probs, axis=0)
all_predictive_entropy = np.array(all_predictive_entropy)
all_aleatoric_entropy = np.array(all_aleatoric_entropy)
all_epistemic_uncertainty = np.array(all_epistemic_uncertainty)

print(
    f"Selesai MC Dropout inference pada {len(all_labels)} gambar test (T={T_MC} sample tiap gambar)."
)

In [ ]:
# @title Classification report, confusion matrix, QWK
IDX_TO_LABEL = {v: k for k, v in DRDataset.LABEL_TO_IDX.items()}
class_names = [IDX_TO_LABEL[i] for i in range(NUM_CLASSES)]

test_acc = accuracy_score(all_labels, all_preds)
test_qwk = cohen_kappa_score(all_labels, all_preds, weights="quadratic")
print(f"Test Accuracy : {test_acc:.4f}")
print(f"Test QWK      : {test_qwk:.4f}\n")

print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    ax=ax,
)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title(f"Confusion Matrix (Test) - QWK={test_qwk:.3f}")
plt.tight_layout()
plt.show()

In [ ]:
# @title Analisis uncertainty: benar vs salah, per kelas, & referral curve
is_correct = all_preds == all_labels

print("Rata-rata predictive entropy:")
print(
    f"  Prediksi BENAR : {all_predictive_entropy[is_correct].mean():.4f} (n={is_correct.sum()})"
)
print(
    f"  Prediksi SALAH : {all_predictive_entropy[~is_correct].mean():.4f} (n={(~is_correct).sum()})"
)

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# (1) distribusi predictive entropy: benar vs salah
axes[0].hist(
    all_predictive_entropy[is_correct], bins=30, alpha=0.6, label="Benar", density=True
)
axes[0].hist(
    all_predictive_entropy[~is_correct], bins=30, alpha=0.6, label="Salah", density=True
)
axes[0].set_xlabel("Predictive entropy")
axes[0].set_ylabel("Density")
axes[0].set_title("Uncertainty: prediksi benar vs salah")
axes[0].legend()

# (2) epistemic uncertainty per kelas asli
df_unc = pd.DataFrame(
    {
        "true_label": [class_names[i] for i in all_labels],
        "epistemic": all_epistemic_uncertainty,
        "aleatoric": all_aleatoric_entropy,
        "correct": is_correct,
    }
)
sns.boxplot(data=df_unc, x="true_label", y="epistemic", order=class_names, ax=axes[1])
axes[1].set_title("Epistemic uncertainty per kelas (true label)")
axes[1].tick_params(axis="x", rotation=30)

# (3) referral / selective-prediction curve
order_idx = np.argsort(-all_predictive_entropy)  # paling tidak yakin lebih dahulu
sorted_correct = is_correct[order_idx]
n = len(sorted_correct)
coverage, sel_acc = [], []
for reject_frac in np.linspace(0, 0.5, 21):
    n_reject = int(reject_frac * n)
    kept = sorted_correct[n_reject:]
    coverage.append(1 - reject_frac)
    sel_acc.append(kept.mean() if len(kept) > 0 else np.nan)

axes[2].plot(coverage, sel_acc, marker="o")
axes[2].set_xlabel("Coverage (fraksi data yang TETAP diprediksi model)")
axes[2].set_ylabel("Accuracy pada data yang di-cover")
axes[2].set_title("Selective prediction / referral curve")
axes[2].invert_xaxis()

plt.tight_layout()
plt.show()

In [ ]:
# @title ROC-AUC
from sklearn.metrics import auc, roc_curve
from sklearn.preprocessing import label_binarize

# Binarize labels untuk ROC per kelas
y_test_bin = label_binarize(all_labels, classes=[0, 1, 2, 3, 4])

# Hitung ROC dan AUC untuk setiap kelas
fpr = dict()
tpr = dict()
roc_auc = dict()
for i in range(NUM_CLASSES):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], all_mean_probs[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Plot ROC Curves
plt.figure(figsize=(10, 8))
colors = ["blue", "red", "green", "orange", "purple"]
for i, color in zip(range(NUM_CLASSES), colors):
    plt.plot(
        fpr[i],
        tpr[i],
        color=color,
        lw=2,
        label=f"ROC {class_names[i]} (AUC = {roc_auc[i]:.2f})",
    )

plt.plot([0, 1], [0, 1], "k--", lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Receiver Operating Characteristic (ROC) per Class")
plt.legend(loc="lower right")
plt.show()

In [ ]:
# @title Standard Deviation dari MC samples untuk mengukur variabilitas (ketidakpastian)
@torch.no_grad()
def get_mc_std(model, loader, T=25):
    enable_mc_dropout(model)
    all_stds = []
    for imgs, _ in tqdm(loader, desc="Calculating MC Std Dev"):
        imgs = imgs.to(DEVICE)
        probs_samples = []
        for _ in range(T):
            probs_samples.append(F.softmax(model(imgs), dim=1).unsqueeze(0))
        probs_samples = torch.cat(probs_samples, dim=0)  # (T, B, C)
        std_dev = probs_samples.std(dim=0).mean(dim=1)  # Rata-rata std dev antar kelas
        all_stds.extend(std_dev.cpu().numpy())
    return np.array(all_stds)


test_stds = get_mc_std(model, test_loader, T=T_MC)

# Visualisasi Korelasi Error vs Uncertainty
plt.figure(figsize=(8, 6))
sns.kdeplot(test_stds[is_correct], label="Correct Predictions", fill=True)
sns.kdeplot(test_stds[~is_correct], label="Incorrect Predictions", fill=True)
plt.title("Distribution of Predictive Standard Deviation (Uncertainty)")
plt.xlabel("Standard Deviation across MC Samples")
plt.ylabel("Density")
plt.legend()
plt.show()